In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
def write_to_silver(
    input_df ,
    target_table,
    merge_condition,
    columns_to_update
):
    input_df = (
        input_df.withColumn("updated_timestamp",F.current_timestamp())
        .withColumn("created_timestamp",F.current_timestamp())
    )
    
    if not spark.catalog.tableExists(target_table):
        (
        input_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
        )

    else :
        delta_table = DeltaTable.forName(spark,target_table)
        updated_map = {column: f"s.{column}" for column in columns_to_update}
        updated_map['updated_timestamp'] = "s.updated_timestamp"
    (
        delta_table.alias("t")
        .merge(
            input_df.alias("s"),
            merge_condition
        )
        .whenMatchedUpdate(
            condition="s.batch_id >= t.batch_id",
            set = updated_map
        )
        .whenNotMatchedInsertAll()
        .execute()
    )


